# 3.1 轻量化架构设计

从架构层面设计更适合端侧部署的模型，使其在参数量更少的情况下达到与大模型可比的性能。核心思想：通过精心设计训练数据和训练策略，使小参数量模型达到甚至超越更大模型的性能。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch version: {torch.__version__}")

---
### 小参数量模型（SLM）设计

研究表明，通过高质量训练数据和知识蒸馏，小参数量模型（1B-3B）可以达到甚至超越更大模型的性能。代表：Phi系列、Gemma-2B、MiniCPM。

关键设计原则：
1. **深而窄**：更多层+更窄的隐藏维度
2. **嵌入共享**：输入输出嵌入共享权重
3. **旋转位置编码（RoPE）**：比学习位置编码更高效

In [ ]:
class RotaryPositionEmbedding(nn.Module):
    """旋转位置编码 (RoPE)"""
    def __init__(self, dim, max_seq_len=512, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        t = torch.arange(max_seq_len).float()
        freqs = torch.outer(t, inv_freq)
        self.register_buffer('cos_cache', freqs.cos())
        self.register_buffer('sin_cache', freqs.sin())

    def forward(self, x, seq_len=None):
        if seq_len is None:
            seq_len = x.shape[2]
        cos = self.cos_cache[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cache[:seq_len].unsqueeze(0).unsqueeze(0)
        x1, x2 = x[..., ::2], x[..., 1::2]
        rotated = torch.stack([-x2, x1], dim=-1).reshape_as(x)
        return x * cos + rotated * sin

class LightweightAttention(nn.Module):
    """轻量GQA注意力 + RoPE"""
    def __init__(self, dim, n_heads, n_kv_heads=None):
        super().__init__()
        n_kv_heads = n_kv_heads or n_heads
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.q_proj = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope = RotaryPositionEmbedding(self.head_dim)

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q = self.rope(q, N)
        k = self.rope(k, N)
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out)

class LightweightBlock(nn.Module):
    """轻量Transformer块"""
    def __init__(self, dim, n_heads, n_kv_heads=None, ffn_mult=4):
        super().__init__()
        self.attn = LightweightAttention(dim, n_heads, n_kv_heads)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * ffn_mult, bias=False),
            nn.SiLU(),
            nn.Linear(dim * ffn_mult, dim, bias=False),
        )
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class SmallLanguageModel(nn.Module):
    """小型语言模型 (SLM)"""
    def __init__(self, vocab_size=32000, dim=512, n_layers=12, n_heads=8,
                 n_kv_heads=2, ffn_mult=4, tied_embeddings=True):
        super().__init__()
        self.tied_embeddings = tied_embeddings
        self.emb = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList([
            LightweightBlock(dim, n_heads, n_kv_heads, ffn_mult)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(dim)
        if not tied_embeddings:
            self.lm_head = nn.Linear(dim, vocab_size, bias=False)

    def forward(self, input_ids):
        h = self.emb(input_ids)
        for layer in self.layers:
            h = layer(h)
        h = self.ln_f(h)
        if self.tied_embeddings:
            logits = F.linear(h, self.emb.weight)
        else:
            logits = self.lm_head(h)
        return logits

configs = {
    'SLM-125M': {'dim': 512, 'n_layers': 12, 'n_heads': 8, 'n_kv_heads': 2},
    'SLM-350M': {'dim': 768, 'n_layers': 18, 'n_heads': 12, 'n_kv_heads': 4},
    'SLM-1B':   {'dim': 1024, 'n_layers': 24, 'n_heads': 16, 'n_kv_heads': 4},
}

print(f"=== SLM架构参数对比 ===")
print(f"{'模型':<12} {'dim':<6} {'layers':<8} {'heads':<7} {'kv_heads':<10} {'参数量':<12} {'嵌入共享节省':<12}")
print("-" * 67)

for name, cfg in configs.items():
    model = SmallLanguageModel(vocab_size=32000, **cfg, tied_embeddings=True)
    params = sum(p.numel() for p in model.parameters())
    emb_params = 32000 * cfg['dim']
    print(f"{name:<12} {cfg['dim']:<6} {cfg['n_layers']:<8} {cfg['n_heads']:<7} {cfg['n_kv_heads']:<10} {params/1e6:.1f}M      {emb_params/1e6:.1f}M")

print(f"\n关键设计:")
print(f"1. GQA(n_kv_heads < n_heads): 减少KV Cache，加速推理")
print(f"2. 嵌入共享: 输入输出共享emb权重，节省{32000*512/1e6:.0f}M参数")
print(f"3. RoPE: 无需学习位置编码，外推性更好")
print(f"4. SiLU激活: 比GELU计算更简单，效果相当")

### 深度与宽度的最优权衡

研究表明，在相同参数预算下，更深更窄的网络比浅更宽的网络更适合语言建模任务。

In [ ]:
def create_transformer(vocab_size, dim, n_layers, n_heads=4):
    """创建指定配置的Transformer"""
    model = SmallLanguageModel(
        vocab_size=vocab_size, dim=dim, n_layers=n_layers,
        n_heads=n_heads, n_kv_heads=2, tied_embeddings=True
    )
    return model

vocab_size = 5000
target_params = 2_000_000

configs = [
    ('浅宽', 256, 4),
    ('中等', 192, 8),
    ('深窄', 128, 16),
    ('极深窄', 96, 24),
]

print(f"=== 深度 vs 宽度权衡 (目标~{target_params/1e6:.0f}M参数) ===")
print(f"{'配置':<10} {'dim':<6} {'layers':<8} {'参数量':<12} {'每层参数':<12}")
print("-" * 48)

for name, dim, n_layers in configs:
    model = create_transformer(vocab_size, dim, n_layers)
    params = sum(p.numel() for p in model.parameters())
    layer_params = params / n_layers
    print(f"{name:<10} {dim:<6} {n_layers:<8} {params/1e6:.2f}M      {layer_params/1e3:.1f}K")

x = torch.randint(0, vocab_size, (2, 32))
results = {}
for name, dim, n_layers in configs:
    model = create_transformer(vocab_size, dim, n_layers)
    params = sum(p.numel() for p in model.parameters())
    with torch.no_grad():
        start = time.time()
        for _ in range(10):
            out = model(x)
        elapsed = time.time() - start
    results[name] = {'params': params, 'time': elapsed, 'dim': dim, 'layers': n_layers}

print(f"\n推理速度对比 (10次前向):")
for name, r in results.items():
    print(f"  {name}: {r['time']:.4f}s ({r['dim']}x{r['layers']})")

print(f"\n结论: 深窄模型在相同参数量下通常表现更好，但推理延迟更高")
print(f"端侧部署需在精度和延迟间权衡，通常选择中等深度")